# General set-up

In [1]:
import ee
import os
import pandas as pd

In [ ]:
# 1. Inicializar Google Earth Engine

import ee
ee.Authenticate()
ee.Initialize()


In [ ]:
# Coordenadas del punto central (Aoulouz, Marruecos)
LON = -8.2727195
LAT = 30.639084
BUFFER_RADIUS_KM = 16

# Crear el área de interés (AOI)
point = ee.Geometry.Point([LON, LAT])
aoi = point.buffer(BUFFER_RADIUS_KM * 1000)  # Buffer en metros

# Rango de años y meses
START_YEAR = 1984
END_YEAR = 2010
MONTHS = [1,2,3,4,8,9,10,11,12]  


# NDVI - Landsat 5

### **Objective**
To use vegetation greenness as a proxy for agricultural productivity, measured through the Normalized Difference Vegetation Index (NDVI), in order to assess the impact of hydraulic infrastructure (dam) and institutional reforms. NDVI captures spatial and temporal variation in crop vigor at the field and parcel level, complementing precipitation as a control for climatic conditions.

---

### **Choice of Data Source:** Landsat 5 Thematic Mapper (TM)
- **Temporal coverage:** 1984–2010, fully covering the study period.
- **Spatial resolution:** 30 m for reflective bands.
- **Long-term consistency:** Landsat 5 provides a stable and continuous time series suitable for historical analysis.
- **Spectral relevance:** NDVI is computed using the red (Band 3) and near-infrared (Band 4) bands, which are specifically designed to capture vegetation reflectance properties.

---

### **NDVI Definition**
NDVI is defined as:

\[
NDVI = \frac{NIR - Red}{NIR + Red}
\]

where:
- **NIR** corresponds to Landsat 5 TM Band 4  
- **Red** corresponds to Landsat 5 TM Band 3  

Healthy vegetation absorbs red light and strongly reflects near-infrared radiation, resulting in higher NDVI values. NDVI theoretically ranges from −1 to +1, with higher values indicating denser and more vigorous vegetation.

---

### **Processing Method:** Aggregation at 200m × 200m Grid Level
NDVI is computed at the native 30 m resolution from cloud-masked Landsat surface reflectance imagery. 

Processing steps include:
- Cloud and cloud-shadow masking using Landsat quality assurance (QA) bands.
- Computation of NDVI at 30 m resolution.
- Temporal aggregation aligned with the relevant agricultural season.

---


In [ ]:
# Colección de Landsat 5 
L5_COLLECTION = 'LANDSAT/LT05/C02/T1_L2'  # Surface Reflectance Collection 2

# Bandas de interés para el NDVI
# En Landsat 5 L2: SR_B4 (NIR) y SR_B3 (Red) para NDVI
BANDS = ['SR_B3', 'SR_B4']  # Red y NIR para NDVI

# Nombre de la carpeta en Google Drive
DRIVE_FOLDER_NAME = 'GEE_Landsat5_Exports_Aoulouz_1991_2000_new_months'

In [7]:
# --- Funciones Auxiliares ---

def maskL5Clouds(image):
    """
    Función mejorada para enmascarar nubes usando QA_PIXEL band
    """
    qa_band = image.select('QA_PIXEL')
    
    # Bits para diferentes condiciones en Collection 2
    DILATED_CLOUD_BIT = 1
    CLOUD_BIT = 3
    CLOUD_SHADOW_BIT = 4
    SNOW_BIT = 5
    WATER_BIT = 7
    
    # Crear máscaras
    dilated_cloud_mask = qa_band.bitwiseAnd(1 << DILATED_CLOUD_BIT).eq(0)
    cloud_mask = qa_band.bitwiseAnd(1 << CLOUD_BIT).eq(0)
    cloud_shadow_mask = qa_band.bitwiseAnd(1 << CLOUD_SHADOW_BIT).eq(0)
    snow_mask = qa_band.bitwiseAnd(1 << SNOW_BIT).eq(0)
    water_mask = qa_band.bitwiseAnd(1 << WATER_BIT).eq(0)
    
    # Combinar todas las máscaras
    combined_mask = (dilated_cloud_mask
                    .And(cloud_mask)
                    .And(cloud_shadow_mask)
                    .And(snow_mask)
                    .And(water_mask))
    
    # Aplicar factores de escala para Surface Reflectance
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    
    return image.addBands(optical_bands, None, True).updateMask(combined_mask)

def get_days_in_month(year, month):
    """Helper function to get the number of days in a month"""
    if month in [1, 3, 5, 7, 8, 10, 12]:
        return 31
    elif month in [4, 6, 9, 11]:
        return 30
    else:  # February
        if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):
            return 29
        else:
            return 28


In [ ]:
# --- Proceso Principal ---

print(f"Área de interés creada: Buffer de {BUFFER_RADIUS_KM} km alrededor de ({LAT}, {LON})")
print(f"Procesando años {START_YEAR}-{END_YEAR}, meses {MONTHS}")
print(f"Usando colección: {L5_COLLECTION}")
print(f"Bandas seleccionadas: {BANDS} (Red y NIR para NDVI)")
print("-" * 50)

# Bucle sobre cada año y mes
for year in range(START_YEAR, END_YEAR + 1):
    for month in MONTHS:
        print(f"Procesando Año: {year}, Mes: {month}")
        
        # Crear fechas de inicio y fin
        start_date = f'{year}-{month:02d}-01'
        end_date = f'{year}-{month:02d}-{get_days_in_month(year, month):02d}'
        
        # Filtrar la colección
        collection_filtered = (ee.ImageCollection(L5_COLLECTION)
                             .filterDate(start_date, end_date)
                             .filterBounds(aoi)
                             .map(maskL5Clouds)
                             .select(BANDS))
        
        # Verificar si hay imágenes disponibles
        try:
            collection_size = collection_filtered.size().getInfo()
            
            if collection_size > 0:
                print(f"  Encontradas {collection_size} imágenes")
                
                # Crear el compuesto de la media
                image_composite = collection_filtered.mean()
                
                # Definir el nombre del archivo
                file_name = f'Aoulouz_L5_RedNIR_{year}_{month:02d}'
                
                # Exportar la imagen
                task = ee.batch.Export.image.toDrive(
                    image=image_composite.clip(aoi),
                    description=file_name,
                    folder=DRIVE_FOLDER_NAME,
                    fileNamePrefix=file_name,
                    scale=30,  # Resolución de Landsat 5
                    region=aoi.getInfo()['coordinates'],
                    maxPixels=1e10,
                    crs='EPSG:4326'  # Añadido sistema de coordenadas
                )
                
                task.start()
                print(f"  ✓ Tarea de exportación iniciada: {file_name}")
                
            else:
                print(f"  ✗ No se encontraron imágenes para {year}-{month:02d}")
                
        except Exception as e:
            print(f"  ✗ Error procesando {year}-{month:02d}: {e}")



print("- Todas las tareas de exportación han sido iniciadas")
print("- Progreso en: https://code.earthengine.google.com/tasks")

Área de interés creada: Buffer de 16 km alrededor de (30.639084, -8.2727195)
Procesando años 1984-2010, meses [1, 2, 3, 4, 8, 9, 10, 11, 12]
Usando colección: LANDSAT/LT05/C02/T1_L2
Bandas seleccionadas: ['SR_B3', 'SR_B4'] (Red y NIR para NDVI)
--------------------------------------------------
Procesando Año: 1984, Mes: 1
  ✗ No se encontraron imágenes para 1984-01
Procesando Año: 1984, Mes: 2
  ✗ No se encontraron imágenes para 1984-02
Procesando Año: 1984, Mes: 3
  ✗ No se encontraron imágenes para 1984-03
Procesando Año: 1984, Mes: 4
  Encontradas 1 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_RedNIR_1984_04
Procesando Año: 1984, Mes: 8
  Encontradas 1 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_RedNIR_1984_08
Procesando Año: 1984, Mes: 9
  Encontradas 2 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_RedNIR_1984_09
Procesando Año: 1984, Mes: 10
  Encontradas 2 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_RedNIR_1984_10
Procesando Año: 1984, Me

## Number of pictures

In [ ]:
import pandas as pd

# --- Parámetros para el conteo ---

# Colección de Landsat 5
L5_COLLECTION = 'LANDSAT/LT05/C02/T1_L2'

# Lista para almacenar los resultados
image_counts = []

print("Iniciando el conteo de imágenes Landsat 5 por mes...")

# --- Bucle para contar las imágenes ---

for year in range(START_YEAR, END_YEAR + 1):
    for month in MONTHS:
        # Definir el rango de fechas para el mes actual
        start_date = ee.Date.fromYMD(year, month, 1)
        end_date = start_date.advance(1, 'month')

        # Filtrar la colección y obtener el número de imágenes
        count = (ee.ImageCollection(L5_COLLECTION)
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .size()
                 .getInfo())
        
        # Guardar el resultado
        image_counts.append({
            'Año': year,
            'Mes': month,
            'Num_Imagenes': count
        })

print("Conteo finalizado.")

# --- Presentar los resultados en una tabla ---

# Convertir la lista de resultados a un DataFrame de Pandas
df_counts = pd.DataFrame(image_counts)

Iniciando el conteo de imágenes Landsat 5 por mes...
Conteo finalizado.


In [14]:
df_counts

,Año,Mes,Num_Imagenes
0,1984,5,2
1,1984,6,2
2,1984,7,0
3,1985,5,1
4,1985,6,1
...,...,...,...
82,2011,6,0
83,2011,7,0
84,2012,5,0
85,2012,6,0


In [15]:
df_counts.to_csv("landsat_5.csv")

# Precipitation

**Objective**
To use precipitation as a control variable to isolate the causal impact of hydraulic infrastructure (dam) and institutional reforms on agricultural productivity (NDVI/NDWI), separating it from natural climate variability.

**Choice of Data Source:** CHIRPS
- Temporal coverage: Daily data available since 1981, covering the full study period (1984–2010).
- Spatial resolution: ~5.5 km (0.05 degrees).
- Designed for data-scarce regions: CHIRPS is tailored for arid and semi-arid zones by combining satellite observations with in-situ station data.

**Processing Method:** Aggregation by Area of Interest (AOI)
Instead of extracting precipitation per 200m × 200m polygon (grid cell), we compute the average precipitation across the entire Area of Interest (AOI) — a 16 km radius — and export it as a single CSV file.

**Resolution:**

|     Grid resolution    |       CHIRPS resolution      |
|:----------------------:|:----------------------------:|
| 200m × 200m (0.04 km²) | 5.5 km × 5.5 km (~30.25 km²) |

A single CHIRPS pixel is over 750 times larger than a 200m polygon. Precipitation varies at a broader (kilometric) scale, while vegetation indices (NDVI/NDWI) vary at finer scales (field/parcel level). No real precipitation variation exists at the 200m scale in CHIRPS — many 200m pixels fall within the same CHIRPS cell. Extracting per-200m polygon would assign identical or interpolated values across many grid cells, adding no meaningful spatial information.

In [ ]:
# --- Definición del Área de Interés (AOI) ---

LON = -8.2727195
LAT = 30.639084
BUFFER_RADIUS_KM = 16 # Mismo buffer que para la extracción de NDVI/NDWI

point = ee.Geometry.Point([LON, LAT])
aoi = point.buffer(BUFFER_RADIUS_KM * 1000) # Buffer en metros

# --- Definición del Rango Temporal ---
START_YEAR = 1984
END_YEAR = 2010
MONTHS = [1,2,3,4,5,6,7,8,9,10,11,12] 

# Nueva definición para la temporada húmeda previa
# Por ejemplo: de Octubre del año anterior a Abril del año actual
WET_SEASON_MONTHS_START = 10 # Octubre
WET_SEASON_MONTHS_END = 4   # Abril

# Función auxiliar para obtener días en el mes (movida al inicio para ser reutilizable)
def get_days_in_month(year, month):
    if month == 2:
        return 29 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 28
    elif month in [4, 6, 9, 11]:
        return 30
    else:
        return 31

Earth Engine inicializado correctamente.


In [ ]:
print(f"Área: Buffer {BUFFER_RADIUS_KM}km alrededor de ({LAT}, {LON})")
print(f"Periodo de estudio principal: {START_YEAR}-{END_YEAR}")
print(f"Meses de interés para NDVI/NDWI: {MONTHS}")
print(f"Meses de interés para temporada húmeda previa: Octubre (año-1) a Abril (año)")
print("="*60)

# --- Colección de CHIRPS precipitación ---
chirps_collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterDate(f'{START_YEAR - 1}-{WET_SEASON_MONTHS_START:02d}-01', f'{END_YEAR + 1}-01-01') \
    .select('precipitation')

# --- Extracción de Precipitación Anual ---
print("\n1. Procesando Precipitación Anual (total por año)...")
annual_precip_data = []
for year in range(START_YEAR, END_YEAR + 1):
    # Filtrar por año y sumar toda la precipitación diaria para el año completo
    annual_sum_image = chirps_collection.filterDate(f'{year}-01-01', f'{year}-12-31').sum()

    try:
        mean_precip_value = annual_sum_image.reduceRegion(
            reducer=ee.Reducer.mean(), # Media de los píxeles dentro de la región
            geometry=aoi,
            scale=5000, # Resolución de CHIRPS es ~5.5 km
            maxPixels=1e10
        ).get('precipitation').getInfo()

        annual_precip_data.append([year, mean_precip_value])
        print(f"   ✓ {year}: {mean_precip_value:.2f} mm/año")

    except Exception as e:
        print(f"   ✗ Error al procesar precipitación anual para {year}: {e}")
        annual_precip_data.append([year, None])

# Guardar precipitación anual como CSV
df_annual_precip = pd.DataFrame(annual_precip_data, columns=['year', 'annual_total_precip_mm'])
output_filename_annual = 'chirps_annual_precip_Aoulouz.csv'
df_annual_precip.to_csv(output_filename_annual, index=False)


DESCARGA DE DATOS DE PRECIPITACIÓN CHIRPS
Área: Buffer 16km alrededor de (30.639084, -8.2727195)
Periodo de estudio principal: 1984-2010
Meses de interés para NDVI/NDWI: [5, 6, 7]
Meses de interés para temporada húmeda previa: Octubre (año-1) a Abril (año)

1. Procesando Precipitación Anual (total por año)...
   ✓ 1984: 228.58 mm/año
   ✓ 1985: 246.79 mm/año
   ✓ 1986: 200.43 mm/año
   ✓ 1987: 316.89 mm/año
   ✓ 1988: 351.43 mm/año
   ✓ 1989: 372.47 mm/año
   ✓ 1990: 206.56 mm/año
   ✓ 1991: 246.17 mm/año
   ✓ 1992: 189.24 mm/año
   ✓ 1993: 266.07 mm/año
   ✓ 1994: 197.68 mm/año
   ✓ 1995: 271.32 mm/año
   ✓ 1996: 396.44 mm/año
   ✓ 1997: 232.25 mm/año
   ✓ 1998: 257.18 mm/año
   ✓ 1999: 302.44 mm/año
   ✓ 2000: 171.54 mm/año
   ✓ 2001: 170.04 mm/año
   ✓ 2002: 345.17 mm/año
   ✓ 2003: 282.40 mm/año
   ✓ 2004: 266.81 mm/año
   ✓ 2005: 260.40 mm/año
   ✓ 2006: 288.81 mm/año
   ✓ 2007: 207.41 mm/año
   ✓ 2008: 239.31 mm/año
   ✓ 2009: 285.97 mm/año
   ✓ 2010: 309.79 mm/año

✓ Precipitaci

In [14]:
# --- Extracción de Precipitación Mensual de la Temporada de Crecimiento ---
print("\n2. Procesando Precipitación Mensual (total por mes en temporada de crecimiento)...")
monthly_growing_season_precip_data = []
for year in range(START_YEAR, END_YEAR + 1):
    for month in MONTHS: # Se usa la variable MONTHS que ya tienes definida
        days_in_month = get_days_in_month(year, month)
        start_date = f'{year}-{month:02d}-01'
        end_date = f'{year}-{month:02d}-{days_in_month:02d}'

        # Sumar toda la precipitación diaria para el mes actual
        monthly_sum_image = chirps_collection.filterDate(start_date, end_date).sum()

        try:
            # Reducir la suma mensual a un valor medio para toda la región AOI
            mean_precip_value = monthly_sum_image.reduceRegion(
                reducer=ee.Reducer.mean(), # Media de los píxeles dentro de la región
                geometry=aoi,
                scale=5000,
                maxPixels=1e10
            ).get('precipitation').getInfo()

            monthly_growing_season_precip_data.append([year, month, mean_precip_value])
            print(f"   ✓ {year}-{month:02d}: {mean_precip_value:.2f} mm/mes (Temporada Crecimiento)")

        except Exception as e:
            print(f"   ✗ Error al procesar precipitación mensual para {year}-{month:02d}: {e}")
            monthly_growing_season_precip_data.append([year, month, None])

# Guardar precipitación mensual como CSV
df_monthly_growing_season_precip = pd.DataFrame(monthly_growing_season_precip_data, columns=['year', 'month', 'growing_season_monthly_total_precip_mm'])
output_filename_monthly_growing = 'chirps_growing_season_monthly_precip_Aoulouz.csv'
df_monthly_growing_season_precip.to_csv(output_filename_monthly_growing, index=False)
print(f"\n✓ Precipitación mensual de la temporada de crecimiento guardada en '{output_filename_monthly_growing}'")


2. Procesando Precipitación Mensual (total por mes en temporada de crecimiento)...
   ✓ 1984-05: 21.89 mm/mes (Temporada Crecimiento)
   ✓ 1984-06: 2.92 mm/mes (Temporada Crecimiento)
   ✓ 1984-07: 0.00 mm/mes (Temporada Crecimiento)
   ✓ 1985-05: 9.07 mm/mes (Temporada Crecimiento)
   ✓ 1985-06: 0.00 mm/mes (Temporada Crecimiento)
   ✓ 1985-07: 0.00 mm/mes (Temporada Crecimiento)
   ✓ 1986-05: 19.97 mm/mes (Temporada Crecimiento)
   ✓ 1986-06: 0.78 mm/mes (Temporada Crecimiento)
   ✓ 1986-07: 1.16 mm/mes (Temporada Crecimiento)
   ✓ 1987-05: 18.35 mm/mes (Temporada Crecimiento)
   ✓ 1987-06: 1.42 mm/mes (Temporada Crecimiento)
   ✓ 1987-07: 1.26 mm/mes (Temporada Crecimiento)
   ✓ 1988-05: 16.92 mm/mes (Temporada Crecimiento)
   ✓ 1988-06: 0.49 mm/mes (Temporada Crecimiento)
   ✓ 1988-07: 0.00 mm/mes (Temporada Crecimiento)
   ✓ 1989-05: 6.60 mm/mes (Temporada Crecimiento)
   ✓ 1989-06: 1.14 mm/mes (Temporada Crecimiento)
   ✓ 1989-07: 2.00 mm/mes (Temporada Crecimiento)
   ✓ 1990-05

In [15]:
# --- NUEVA SECCIÓN: Extracción de Precipitación Total de la Temporada Húmeda Previa ---
print("\n3. Procesando Precipitación Total de la Temporada Húmeda Previa (Octubre del año-1 a Abril del año)...")
wet_season_precip_data = []
# Iteramos para cada AÑO EN EL RANGO DE ESTUDIO (1984-2010),
# la temporada húmeda previa corresponde al año ANTERIOR y al inicio del AÑO ACTUAL.
for year in range(START_YEAR, END_YEAR + 1):
    # La temporada húmeda para el 'año' dado, va desde Octubre del 'año-1' hasta Abril del 'año'.
    start_date_wet_season = f'{year - 1}-{WET_SEASON_MONTHS_START:02d}-01'
    # El final de la temporada húmeda es el último día de Abril del 'año'
    days_in_april = get_days_in_month(year, WET_SEASON_MONTHS_END) # Obtener días de abril para el año actual
    end_date_wet_season = f'{year}-{WET_SEASON_MONTHS_END:02d}-{days_in_april:02d}'

    # Filtrar y sumar la precipitación diaria para esta temporada específica
    wet_season_sum_image = chirps_collection.filterDate(start_date_wet_season, end_date_wet_season).sum()

    try:
        mean_wet_season_precip = wet_season_sum_image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=5000,
            maxPixels=1e10
        ).get('precipitation').getInfo()
        
        wet_season_precip_data.append([year, mean_wet_season_precip])
        print(f"   ✓ {year} (Wet Season): {mean_wet_season_precip:.2f} mm (Oct {year-1} - Abr {year})")

    except Exception as e:
        print(f"   ✗ Error al procesar precipitación de temporada húmeda para {year}: {e}")
        wet_season_precip_data.append([year, None])

df_wet_season_precip = pd.DataFrame(wet_season_precip_data, columns=['year', 'previous_wet_season_total_precip_mm'])
output_filename_wet_season = 'chirps_previous_wet_season_precip_Aoulouz.csv'
df_wet_season_precip.to_csv(output_filename_wet_season, index=False)
print(f"\n✓ Precipitación total de la temporada húmeda previa guardada en '{output_filename_wet_season}'")


print("\n" + "="*60)
print("PROCESAMIENTO DE PRECIPITACIÓN CHIRPS FINALIZADO")
print("Archivos CSV creados con datos agregados para el AOI especificado.")
print("Estos datos son controles cruciales para su análisis de panel.")
print("Considere usar la precipitación de la temporada de crecimiento Y la de la temporada húmeda previa en su modelo.")
print("="*60)


3. Procesando Precipitación Total de la Temporada Húmeda Previa (Octubre del año-1 a Abril del año)...
   ✓ 1984 (Wet Season): 156.15 mm (Oct 1983 - Abr 1984)
   ✓ 1985 (Wet Season): 252.00 mm (Oct 1984 - Abr 1985)
   ✓ 1986 (Wet Season): 183.85 mm (Oct 1985 - Abr 1986)
   ✓ 1987 (Wet Season): 145.51 mm (Oct 1986 - Abr 1987)
   ✓ 1988 (Wet Season): 358.12 mm (Oct 1987 - Abr 1988)
   ✓ 1989 (Wet Season): 285.04 mm (Oct 1988 - Abr 1989)
   ✓ 1990 (Wet Season): 306.77 mm (Oct 1989 - Abr 1990)
   ✓ 1991 (Wet Season): 198.88 mm (Oct 1990 - Abr 1991)
   ✓ 1992 (Wet Season): 202.60 mm (Oct 1991 - Abr 1992)
   ✓ 1993 (Wet Season): 178.62 mm (Oct 1992 - Abr 1993)
   ✓ 1994 (Wet Season): 238.53 mm (Oct 1993 - Abr 1994)
   ✓ 1995 (Wet Season): 220.64 mm (Oct 1994 - Abr 1995)
   ✓ 1996 (Wet Season): 314.42 mm (Oct 1995 - Abr 1996)
   ✓ 1997 (Wet Season): 244.74 mm (Oct 1996 - Abr 1997)
   ✓ 1998 (Wet Season): 244.40 mm (Oct 1997 - Abr 1998)
   ✓ 1999 (Wet Season): 193.91 mm (Oct 1998 - Abr 1999)


## Summary of Invariant Topographic Variables
These variables represent physical characteristics of the terrain that do not change over time, making them crucial as static controls in panel econometric models and for understanding spatial heterogeneity. Their consistency with NDVI data (Landsat/Sentinel) is key for rigorous analysis.
### 1. Elevation
Importance for Research:
Fundamental Control: Elevation is a primary factor influencing natural water access (gravity flow), precipitation distribution (orographic effects), temperature patterns (thermal gradients), and oxygen availability.
Initial Condition: Differences in elevation can explain pre-existing variations in agricultural productivity or the feasibility of certain crops, and how these interact with infrastructure and institutions. For example, higher areas may be more challenging to irrigate or naturally have lower productive potential.
Microclimate: Affects the local microclimate, directly impacting crop phenology and water requirements.
Source of Capture:
Dataset: USGS/SRTMGL1_003 (Shuttle Radar Topography Mission Global 1 arc-second V3).
Sensor: SRTM Mission, operated by NASA and the NGA (National Geospatial-Intelligence Agency). It uses Synthetic Aperture Radar (SAR) technology to measure terrain height from space.
Capture Date: SRTM base data was collected in February 2000.
Original Resolution: Approximately 30 meters per pixel globally (1 arc-second).
Processing:
Extraction: The 'elevation' band is directly extracted from the SRTM dataset in Google Earth Engine.
Clipping: The image is clipped to the Area of Interest (AOI) defined by a 16 km buffer around the specified central point.
Scaling/Resolution: The spatial resolution is maintained at 30 meters, ensuring direct compatibility with Landsat data.
Coordinate Reference System (CRS): Exported in EPSG:4326 (WGS84 latitude/longitude), a standard geographic coordinate system.
### 2. Slope
Importance for Research:
Irrigation Feasibility: Slope is a critical determinant of the technical and economic viability of irrigation systems. Plots with steep slopes are much more difficult and costly to irrigate efficiently, which can limit the adoption of new technologies or the impact of water infrastructure.
Erosion and Drainage: Affects surface runoff rates and soil erosion, impacting soil quality and water retention.
Agricultural Practices: Influences the possibility of agricultural mechanization and the type of soil management.
Source of Capture:
Derived from: The SRTM elevation data (USGS/SRTMGL1_003).
Processing:
Calculation: Calculated using Earth Engine's ee.Terrain.slope() function, which derives the slope from the elevation surface.
Units: The output is in degrees (0 to 90), representing the steepness of the terrain.
Clipping and Scaling: Similar to elevation, it is clipped to the AOI and exported at 30 meters resolution with EPSG:4326.
### 3. Aspect
Importance for Research:
Insolation (Solar Exposure): The aspect of a slope determines the amount of solar radiation a surface receives. In the Northern Hemisphere, south-facing slopes receive more direct sunlight and heat.
Evapotranspiration: Higher insolation can lead to higher evapotranspiration rates, affecting irrigation needs and crop water stress.
Local Microclimate: Influences soil temperature, moisture, and the type of vegetation that can thrive. It can be an explanatory factor for heterogeneity in response to the intervention.
Source of Capture:
Derived from: The SRTM elevation data (USGS/SRTMGL1_003).
Processing:
Calculation: Calculated using Earth Engine's ee.Terrain.aspect() function, which determines the direction of the steepest slope.
Units: The output is in degrees (0 to 360), where 0 and 360 degrees represent North, 90 East, 180 South, and 270 West.
Clipping and Scaling: Similar to the others, it is clipped to the AOI and exported at 30 meters resolution with EPSG:4326.

In [1]:
import ee
import os
ee.Authenticate()
ee.Initialize(project='ee-ifmoreno905905')

In [2]:
LON = -8.2727195
LAT = 30.639084
BUFFER_RADIUS_KM = 16

In [4]:
# Crear AOI idéntico al de Landsat
point = ee.Geometry.Point([LON, LAT])
aoi = point.buffer(BUFFER_RADIUS_KM * 1000)

# Carpeta en Drive
DRIVE_FOLDER_NAME = 'GEE_Topographic_Aoulouz'

In [5]:
# 1. ELEVATION - SRTM 30m (perfecto match con Landsat)
print("1. Procesando ELEVATION (SRTM 30m)...")
srtm = ee.Image('USGS/SRTMGL1_003')
elevation = srtm.select('elevation').clip(aoi)

# Exportar elevation
task_elevation = ee.batch.Export.image.toDrive(
    image=elevation,
    description='Aoulouz_Elevation_SRTM30m',
    folder=DRIVE_FOLDER_NAME,
    fileNamePrefix='Aoulouz_Elevation_SRTM30m',
    scale=30,  # MISMA RESOLUCIÓN QUE LANDSAT
    region=aoi.getInfo()['coordinates'],
    maxPixels=1e10,
    crs='EPSG:4326'
)
task_elevation.start()
print("   ✓ Elevation export iniciado")

1. Procesando ELEVATION (SRTM 30m)...
   ✓ Elevation export iniciado


In [6]:
# 2. SLOPE - Calculado desde SRTM
print("2. Procesando SLOPE (derivado de SRTM)...")
slope = ee.Terrain.slope(elevation)  # En grados

# Exportar slope
task_slope = ee.batch.Export.image.toDrive(
    image=slope,
    description='Aoulouz_Slope_SRTM30m',
    folder=DRIVE_FOLDER_NAME,
    fileNamePrefix='Aoulouz_Slope_SRTM30m',
    scale=30,
    region=aoi.getInfo()['coordinates'],
    maxPixels=1e10,
    crs='EPSG:4326'
)
task_slope.start()
print("   ✓ Slope export iniciado")

2. Procesando SLOPE (derivado de SRTM)...
   ✓ Slope export iniciado


In [7]:
# 3. ASPECT - Bonus para robustez (orientación de ladera)
print("3. Procesando ASPECT (orientación de ladera)...")
aspect = ee.Terrain.aspect(elevation)  # En grados (0-360)

task_aspect = ee.batch.Export.image.toDrive(
    image=aspect,
    description='Aoulouz_Aspect_SRTM30m',
    folder=DRIVE_FOLDER_NAME,
    fileNamePrefix='Aoulouz_Aspect_SRTM30m',
    scale=30,
    region=aoi.getInfo()['coordinates'],
    maxPixels=1e10,
    crs='EPSG:4326'
)
task_aspect.start()
print("   ✓ Aspect export iniciado")

3. Procesando ASPECT (orientación de ladera)...
   ✓ Aspect export iniciado


In [8]:
# 4. COMPOSITE TOPOGRÁFICO - Todo en una imagen
print("4. Creando composite topográfico (todas las variables)...")
topo_composite = ee.Image.cat([
    elevation.rename('elevation'),
    slope.rename('slope'),
    aspect.rename('aspect')
])

4. Creando composite topográfico (todas las variables)...


In [9]:
task_composite = ee.batch.Export.image.toDrive(
    image=topo_composite,
    description='Aoulouz_Topographic_Composite',
    folder=DRIVE_FOLDER_NAME,
    fileNamePrefix='Aoulouz_Topographic_Composite',
    scale=30,
    region=aoi.getInfo()['coordinates'],
    maxPixels=1e10,
    crs='EPSG:4326'
)
task_composite.start()
print("   ✓ Topographic composite export iniciado")

print("="*60)
print("DESCARGA DE VARIABLES TOPOGRÁFICAS INVARIANTES")
print(f"Área: Buffer {BUFFER_RADIUS_KM}km alrededor de ({LAT}, {LON})")
print("="*60)

print("\n" + "="*60)
print("RESUMEN VARIABLES TOPOGRÁFICAS:")
print("✓ Elevation (SRTM 30m) - Control fundamental para identificación")
print("✓ Slope (grados) - Afecta potencial de riego y productividad") 
print("✓ Aspect (orientación) - Para robustez adicional")
print("✓ Composite (todas juntas) - Más eficiente para análisis")
print()
print("IMPORTANCIA PARA IDENTIFICACIÓN CAUSAL:")
print("- Elevation: controla acceso natural al agua")
print("- Slope: afecta feasibilidad técnica del riego")
print("- Invariantes → descargar solo UNA VEZ")
print()
print("Monitorea progreso: https://code.earthengine.google.com/tasks")
print("="*60)

   ✓ Topographic composite export iniciado
DESCARGA DE VARIABLES TOPOGRÁFICAS INVARIANTES
Área: Buffer 16km alrededor de (30.639084, -8.2727195)

RESUMEN VARIABLES TOPOGRÁFICAS:
✓ Elevation (SRTM 30m) - Control fundamental para identificación
✓ Slope (grados) - Afecta potencial de riego y productividad
✓ Aspect (orientación) - Para robustez adicional
✓ Composite (todas juntas) - Más eficiente para análisis

IMPORTANCIA PARA IDENTIFICACIÓN CAUSAL:
- Elevation: controla acceso natural al agua
- Slope: afecta feasibilidad técnica del riego
- Invariantes → descargar solo UNA VEZ

Monitorea progreso: https://code.earthengine.google.com/tasks


# NDWI & AVI

In [ ]:
import ee
import os

In [11]:
# --- Configuración del Área de Interés (AOI) y Período ---
LON = -8.2727195
LAT = 30.639084
BUFFER_RADIUS_KM = 16

point = ee.Geometry.Point([LON, LAT])
aoi = point.buffer(BUFFER_RADIUS_KM * 1000)

# Rango de años y meses ajustado
START_YEAR = 1984
END_YEAR = 2010
MONTHS = [5, 6, 7] # Mayo, Junio, Julio (temporada de crecimiento/riego clave)

L5_COLLECTION = 'LANDSAT/LT05/C02/T1_L2' # Surface Reflectance Collection 2

In [12]:
# Bandas de interés para NDWI y AVI:
# SR_B3: Red
# SR_B4: Near Infrared (NIR)
# SR_B5: Shortwave Infrared 1 (SWIR1) - Útil para algunas formulaciones de AVI y análisis de humedad
# SR_B7: Shortwave Infrared 2 (SWIR2) - Crucial para NDWI (Gao)
BANDS = ['SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']

# Nombre de la carpeta en Google Drive (ajustado para reflejar el rango de años y bandas)
DRIVE_FOLDER_NAME = 'GEE_Landsat5_NDWI_AVI_Bands_1984_2010'

In [13]:
# --- Funciones Esenciales para Pre-procesamiento y Fechas ---
def maskL5Clouds(image):
    """
    Función para enmascarar nubes y sombras de nubes, y aplicar factores de escala
    a las bandas de Reflectancia Superficial de Landsat 5 Collection 2.
    """
    qa_band = image.select('QA_PIXEL')

    # Bits relevantes de QA_PIXEL para condiciones no deseadas:
    DILATED_CLOUD_BIT = 1
    CLOUD_BIT = 3
    CLOUD_SHADOW_BIT = 4
    SNOW_BIT = 5
    WATER_BIT = 7 # Se enmascara el agua por defecto para análisis de vegetación/suelo

    # Crear máscaras donde los bits correspondientes no estén configurados (es decir, sea "claro")
    dilated_cloud_mask = qa_band.bitwiseAnd(1 << DILATED_CLOUD_BIT).eq(0)
    cloud_mask = qa_band.bitwiseAnd(1 << CLOUD_BIT).eq(0)
    cloud_shadow_mask = qa_band.bitwiseAnd(1 << CLOUD_SHADOW_BIT).eq(0)
    snow_mask = qa_band.bitwiseAnd(1 << SNOW_BIT).eq(0)
    water_mask = qa_band.bitwiseAnd(1 << WATER_BIT).eq(0)

    # Combinar todas las máscaras para obtener una máscara final de "píxeles buenos"
    combined_mask = (dilated_cloud_mask
                    .And(cloud_mask)
                    .And(cloud_shadow_mask)
                    .And(snow_mask)
                    .And(water_mask))

    # Aplicar factores de escala a las bandas de Reflectancia Superficial (SR_B*)
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)

    # Reemplazar las bandas originales con las escaladas y aplicar la máscara combinada
    return image.addBands(optical_bands, None, True).updateMask(combined_mask)

def get_days_in_month(year, month):
    """Función auxiliar para obtener el número de días en un mes, considerando años bisiestos."""
    if month in [1, 3, 5, 7, 8, 10, 12]:
        return 31
    elif month in [4, 6, 9, 11]:
        return 30
    else: # Febrero
        if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):
            return 29
        else:
            return 28

In [14]:
# --- Proceso Principal ---
print(f"Área de interés creada: Buffer de {BUFFER_RADIUS_KM} km alrededor de ({LAT}, {LON})")
print(f"Procesando años {START_YEAR}-{END_YEAR}, meses {MONTHS}")
print(f"Usando colección: {L5_COLLECTION}")
print(f"Bandas seleccionadas para exportación (para NDWI y AVI): {BANDS}")
print("-" * 50)

for year in range(START_YEAR, END_YEAR + 1):
    for month in MONTHS:
        print(f"Procesando Año: {year}, Mes: {month}")

        start_date = f'{year}-{month:02d}-01'
        end_date = f'{year}-{month:02d}-{get_days_in_month(year, month):02d}'

        collection_filtered = (ee.ImageCollection(L5_COLLECTION)
                             .filterDate(start_date, end_date)
                             .filterBounds(aoi)
                             .map(maskL5Clouds)
                             .select(BANDS)) # Selecciona solo las bandas necesarias

        try:
            collection_size = collection_filtered.size().getInfo()

            if collection_size > 0:
                print(f"  Encontradas {collection_size} imágenes")

                # Crear el compuesto de la media mensual de las bandas seleccionadas
                image_composite = collection_filtered.mean()

                file_name = f'Aoulouz_L5_NDWI_AVI_Bands_{year}_{month:02d}'

                task = ee.batch.Export.image.toDrive(
                    image=image_composite.clip(aoi),
                    description=file_name,
                    folder=DRIVE_FOLDER_NAME,
                    fileNamePrefix=file_name,
                    scale=30,
                    region=aoi, # Se puede pasar directamente el objeto ee.Geometry
                    maxPixels=1e10,
                    crs='EPSG:4326'
                )

                task.start()
                print(f"  ✓ Tarea de exportación iniciada: {file_name}")

            else:
                print(f"  ✗ No se encontraron imágenes para {year}-{month:02d}")

        except Exception as e:
            print(f"  ✗ Error procesando {year}-{month:02d}: {e}")

print("\n" + "="*50)
print("RESUMEN DE EXPORTACIÓN:")
print("- Todas las tareas de exportación de bandas de Landsat 5 han sido iniciadas.")
print("- Se exportan las bandas: SR_B3 (Red), SR_B4 (NIR), SR_B5 (SWIR1), SR_B7 (SWIR2).")
print("- Estas bandas permitirán el cálculo de los siguientes índices en tu entorno local (ej. Python/R):")
print("  - **NDWI (Normalized Difference Water Index - Gao, 1996):** (NIR - SWIR2) / (NIR + SWIR2) => (SR_B4 - SR_B7) / (SR_B4 + SR_B7)")
print("  - **AVI (Advanced Vegetation Index):** Existen varias formulaciones. Con estas bandas puedes calcular:")
print("    - Variantes que usan Red (SR_B3) y NIR (SR_B4), o Red, NIR, y SWIR1 (SR_B5) para realzar la vegetación escasa y diferenciar del suelo.")
print("    - Ejemplo de una formulación común de AVI (adaptable a estas bandas) podría ser:")
print("      `((NIR + 1) * (256 - Red) * (NIR - Red))^(1/3)` (con SR_B4 y SR_B3)")
print("      Asegúrate de consultar la literatura específica para la formulación de AVI que mejor se adapte a tu contexto y datos.")
print("- Monitorea el progreso en: https://code.earthengine.google.com/tasks")
print(f"- Los archivos se guardarán en la carpeta '{DRIVE_FOLDER_NAME}' en Google Drive.")
print("- Las imágenes tendrán resolución de 30m y estarán recortadas al área de interés.")
print("="*50)

Área de interés creada: Buffer de 16 km alrededor de (30.639084, -8.2727195)
Procesando años 1984-2010, meses [5, 6, 7]
Usando colección: LANDSAT/LT05/C02/T1_L2
Bandas seleccionadas para exportación (para NDWI y AVI): ['SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
--------------------------------------------------
Procesando Año: 1984, Mes: 5
  Encontradas 2 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_NDWI_AVI_Bands_1984_05
Procesando Año: 1984, Mes: 6
  Encontradas 2 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_NDWI_AVI_Bands_1984_06
Procesando Año: 1984, Mes: 7
  ✗ No se encontraron imágenes para 1984-07
Procesando Año: 1985, Mes: 5
  Encontradas 1 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_NDWI_AVI_Bands_1985_05
Procesando Año: 1985, Mes: 6
  Encontradas 1 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_NDWI_AVI_Bands_1985_06
Procesando Año: 1985, Mes: 7
  Encontradas 1 imágenes
  ✓ Tarea de exportación iniciada: Aoulouz_L5_NDWI_AVI_Bands_1985_07
Proces

| Year          | Event                                                                                                                                                                                               |
| ------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **1960s**     | Beginning of large-scale agricultural exploitation in the Souss Valley. The groundwater level starts to decline.                                                                                    |
| **1991**      | Construction of the **Aoulouz dam** (capacity: 110 million m³) with three goals: drinking water for Taroudant, public irrigation in the hills, and aquifer recharge.                                |
| **1995**      | The government, supported by King Mohamed VI and the **IFC (World Bank)**, launches a **public-private partnership (PPP)** irrigation project.                                                      |
| **1998–1999** | Construction begins on the **Mokhtar Soussi dam**, part of the PPP project.                                                                                                                         |
| **2002**      | The Mokhtar Soussi dam becomes operational after delays caused by resettlement protests. Its water is channeled to Aoulouz and then to El Guerdane through a 300 km network.                        |
| **2004**      | The IFC declares this PPP the **first of its kind in the world** regarding private water management for irrigation.                                                                                 |
| **2009**      | A **"aquifer contract"** is signed between the regional council and ABHSM. It is participatory and voluntary, promoting sustainable water management. It includes bans on expanding irrigated land. |
| **2011**      | ABHSM estimates an **annual water deficit** of 233 million m³. Groundwater levels have dropped between **30 and 75 meters**. Despite restrictions, irrigated areas continue to expand.              |


# NDWI & CRISP

In [13]:
import ee

# Forzar nueva autenticación con force=True
ee.Authenticate(
    force=True,  # ← Esto ignora credenciales existentes
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.full_control',
        'https://www.googleapis.com/auth/drive'
    ]
)

ee.Initialize()
ee.Initialize(project='ee-ifmoreno905905')

Enter verification code:  4/1Ab32j92FvldDea6FLfEidsrPJ2LGvWGN_wIKfNde4oYpKocL9y1tqXsNbIg



Successfully saved authorization token.


In [14]:
# =============================================================================
# CONFIGURACIÓN
# =============================================================================

CONFIG = {
    'LON': -8.2727195,
    'LAT': 30.639084,
    'BUFFER_RADIUS_KM': 16,
    'START_YEAR': 1984,
    'END_YEAR': 2010,
    'L5_COLLECTION': 'LANDSAT/LT05/C02/T1_L2',
    'CHIRPS_COLLECTION': 'UCSB-CHG/CHIRPS/DAILY',
    'NDWI_FOLDER': 'GEE_NDWI_Vegetation_1984_2010_FINAL',
    'PRECIP_FOLDER': 'GEE_CHIRPS_Monthly_1984_2010_FINAL'
}

point = ee.Geometry.Point([CONFIG['LON'], CONFIG['LAT']])
aoi = point.buffer(CONFIG['BUFFER_RADIUS_KM'] * 1000)

# =============================================================================
# FUNCIONES AUXILIARES
# =============================================================================

def maskL5sr(image):
    """Enmascara nubes, sombras, nieve y agua usando QA_PIXEL y aplica factores de escala."""
    qa_band = image.select('QA_PIXEL')
    cloud_shadow_snow_water_mask = (qa_band.bitwiseAnd(1 << 1).eq(0)
                                    .And(qa_band.bitwiseAnd(1 << 3).eq(0))
                                    .And(qa_band.bitwiseAnd(1 << 4).eq(0))
                                    .And(qa_band.bitwiseAnd(1 << 5).eq(0))
                                    .And(qa_band.bitwiseAnd(1 << 7).eq(0)))
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True).updateMask(cloud_shadow_snow_water_mask)

def calculateNDWI_Gao(image):
    """Calcula el NDWI de Gao (1996) para humedad en vegetación."""
    return image.normalizedDifference(['SR_B4', 'SR_B5']).rename('NDWI')

In [15]:
# =============================================================================
# PASO 1 (SERVER-SIDE): CREAR LISTA DE MESES Y VERIFICAR DISPONIBILIDAD
# =============================================================================

print("="*70)
print("Paso 1: Verificando la disponibilidad de datos en el servidor de GEE...")
print("         (Esto realiza UNA sola llamada de red y puede tardar un momento)")
print("="*70)

# SOLUCIÓN: Crear la lista en Python y convertirla a ee.List
year_month_combinations = []
for year in range(CONFIG['START_YEAR'], CONFIG['END_YEAR'] + 1):
    for month in range(1, 13):
        year_month_combinations.append([year, month])

# Convertir a lista de Earth Engine
year_month_list = ee.List(year_month_combinations)

def check_availability(ym_list):
    """Función server-side para verificar si hay imágenes en un mes dado."""
    ym_list = ee.List(ym_list)
    year = ee.Number(ym_list.get(0))
    month = ee.Number(ym_list.get(1))
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    
    # Comprobar disponibilidad en ambas colecciones
    l5_size = ee.ImageCollection(CONFIG['L5_COLLECTION']).filterDate(start_date, end_date).filterBounds(aoi).size()
    chirps_size = ee.ImageCollection(CONFIG['CHIRPS_COLLECTION']).filterDate(start_date, end_date).filterBounds(aoi).size()
    
    return ee.Dictionary({
        'year': year,
        'month': month,
        'has_l5_data': l5_size.gt(0),
        'has_chirps_data': chirps_size.gt(0)
    })

# ÚNICA LLAMADA A GETINFO para traer la lista de disponibilidad al cliente
availability_list = year_month_list.map(check_availability).getInfo()

# Filtrar en Python solo los meses que tienen datos
months_with_l5 = [item for item in availability_list if item['has_l5_data']]
months_with_chirps = [item for item in availability_list if item['has_chirps_data']]

print(f"✓ Verificación completada.")
print(f"  - Meses con datos Landsat 5: {len(months_with_l5)} de {len(availability_list)}")
print(f"  - Meses con datos CHIRPS: {len(months_with_chirps)} de {len(availability_list)}")

Paso 1: Verificando la disponibilidad de datos en el servidor de GEE...
         (Esto realiza UNA sola llamada de red y puede tardar un momento)
✓ Verificación completada.
  - Meses con datos Landsat 5: 256 de 324
  - Meses con datos CHIRPS: 324 de 324


In [16]:
# =============================================================================
# PASO 2 (CLIENT-SIDE): ITERAR E INICIAR TAREAS DE EXPORTACIÓN
# =============================================================================

# --- EXPORTACIÓN DE NDWI ---
print("\n" + "="*70)
print(f"Paso 2a: Iniciando {len(months_with_l5)} tareas de exportación para NDWI...")
print("="*70)

# Obtener CRS nativo una sola vez para eficiencia
native_crs = ee.ImageCollection(CONFIG['L5_COLLECTION']).filterBounds(aoi).first().select('SR_B1').projection().crs().getInfo()
print(f"Usando CRS nativo para Landsat: {native_crs}")

for item in months_with_l5:
    year = int(item['year'])
    month = int(item['month'])
    
    # Definir la computación de la imagen (esto es server-side)
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    
    collection = (ee.ImageCollection(CONFIG['L5_COLLECTION'])
                    .filterDate(start_date, end_date)
                    .filterBounds(aoi)
                    .map(maskL5sr))
    
    ndwi_composite = collection.map(calculateNDWI_Gao).median()
    
    # Definir la tarea de exportación (esto es client-side)
    file_name = f'Aoulouz_NDWI_{year}_{month:02d}'
    
    task = ee.batch.Export.image.toDrive(
        image=ndwi_composite.clip(aoi).toFloat(),
        description=file_name,
        folder=CONFIG['NDWI_FOLDER'],
        fileNamePrefix=file_name,
        scale=30,
        region=aoi,
        maxPixels=1e10,
        crs=native_crs
    )
    
    # Iniciar la tarea (client-side)
    task.start()
    print(f"  ✓ Tarea NDWI iniciada: {file_name}")


Paso 2a: Iniciando 256 tareas de exportación para NDWI...
Usando CRS nativo para Landsat: EPSG:32629
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_04
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_05
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_06
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_08
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_09
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_10
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_11
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1984_12
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_02
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_03
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_04
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_05
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_06
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_07
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_08
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_09
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_10
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_11
  ✓ Tarea NDWI iniciada: Aoulouz_NDWI_1985_12
  ✓ Tarea NDWI iniciada:

In [18]:
# --- EXPORTACIÓN DE PRECIPITACIÓN ---
print("\n" + "="*70)
print(f"Paso 2b: Iniciando {len(months_with_chirps)} tareas de exportación para Precipitación...")
print("="*70)

for item in months_with_chirps:
    year = int(item['year'])
    month = int(item['month'])
    
    # Definir la computación de la imagen (server-side)
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')
    
    precip_monthly = (ee.ImageCollection(CONFIG['CHIRPS_COLLECTION'])
                      .filterDate(start_date, end_date)
                      .filterBounds(aoi)
                      .select('precipitation')
                      .sum())

    # Definir la tarea de exportación (client-side)
    file_name = f'Aoulouz_Precip_{year}_{month:02d}'
    
    task = ee.batch.Export.image.toDrive(
        image=precip_monthly.clip(aoi).toFloat(),
        description=file_name,
        folder=CONFIG['PRECIP_FOLDER'],
        fileNamePrefix=file_name,
        scale=5566,
        region=aoi,
        maxPixels=1e10,
        crs='EPSG:4326'
    )
    
    # Iniciar la tarea (client-side)
    task.start()
    print(f"  ✓ Tarea Precipitación iniciada: {file_name}")



Paso 2b: Iniciando 324 tareas de exportación para Precipitación...
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_01
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_02
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_03
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_04
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_05
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_06
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_07
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_08
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_09
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_10
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_11
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1984_12
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1985_01
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1985_02
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1985_03
  ✓ Tarea Precipitación iniciada: Aoulouz_Precip_1985_04
  ✓ Tarea Precipitac


KeyboardInterrupt



In [19]:
# =============================================================================
# RESUMEN FINAL
# =============================================================================

print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)
print("✓ Todas las tareas de exportación han sido iniciadas.")
print(f"  - Tareas NDWI: {len(months_with_l5)} -> Carpeta '{CONFIG['NDWI_FOLDER']}'")
print(f"  - Tareas Precipitación: {len(months_with_chirps)} -> Carpeta '{CONFIG['PRECIP_FOLDER']}'")
print("\nArquitectura utilizada:")
print("  - Híbrida (Server + Client): Se minimizan las llamadas de red a una sola para máxima eficiencia.")
print("\nMonitorea el progreso de las tareas en:")
print("  https://code.earthengine.google.com/tasks")
print("="*70)


RESUMEN FINAL
✓ Todas las tareas de exportación han sido iniciadas.
  - Tareas NDWI: 256 -> Carpeta 'GEE_NDWI_Vegetation_1984_2010_FINAL'
  - Tareas Precipitación: 324 -> Carpeta 'GEE_CHIRPS_Monthly_1984_2010_FINAL'

Arquitectura utilizada:
  - Híbrida (Server + Client): Se minimizan las llamadas de red a una sola para máxima eficiencia.

Monitorea el progreso de las tareas en:
  https://code.earthengine.google.com/tasks
